# 2026 MediaEval NewsImages - Image Generation Workflow

**Prompt template:** `news image thumbnail, illustration, vector art, {headline}`

This notebook contains only the image generation workflow used to produce the submitted thumbnails. All experiment and analysis code has been removed.


## Cell 1: Install packages

**Purpose:** Install all Python packages required by this notebook.

| Package | Use |
|---|---|
| `diffusers` | HuggingFace diffusion model framework, drives SDXL-Turbo image generation |
| `transformers` | Loads the CLIP / CFT-CLIP models |
| `open_clip_torch` | OpenCLIP implementation |
| `easyocr` | OCR text detection, used to reject thumbnails that contain rendered text |
| `ftfy` | Fixes broken Unicode encoding in article titles |


In [ ]:
# @title packages
!pip install -q diffusers transformers datasets accelerate safetensors \
    sentencepiece open_clip_torch torch torchvision easyocr ftfy


## Cell 2: Initialize models

**Purpose:** Log in to the HuggingFace Hub and load the core models used by this notebook.

**Key variables:**

| Variable | Description |
|---|---|
| `DEVICE` | Auto-detects `'cuda'` (GPU) or `'cpu'`; affects all tensor operations |
| `pipe` | SDXL-Turbo image generation pipeline (1-step fast diffusion) - the only image generator in this notebook |
| `reader` | EasyOCR reader, detects whether a thumbnail contains visible text |
| `standard_clip_model` | OpenAI CLIP ViT-B/32, baseline image-text similarity scorer |
| `cft_clip_model` | CFT-CLIP (news-thumbnail-specific), used to select the best candidate image |


In [ ]:
# @title setup
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

import torch
import torch.nn.functional as F
import easyocr
from diffusers import AutoPipelineForText2Image
from transformers import AutoModel, AutoProcessor, CLIPProcessor, CLIPModel

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. SDXL-Turbo (1-step image generator)
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16",
).to(DEVICE)

# 2. OCR - rejects thumbnails that contain rendered text
reader = easyocr.Reader(['en'], gpu=(DEVICE == 'cuda'))

# 3. Standard CLIP (baseline selector)
standard_clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
standard_clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
standard_clip_model.eval()

# 4. CFT-CLIP (news-thumbnail-specific selector + embedding source)
cft_clip_processor = AutoProcessor.from_pretrained("humane-lab/CFT-CLIP")
cft_clip_model     = AutoModel.from_pretrained("humane-lab/CFT-CLIP").to(DEVICE)
cft_clip_model.eval()


## Cell 3: Core pipeline functions

**Purpose:** Define the helper functions used by the generation step.

**Key variables and functions:**

| Name | Description |
|---|---|
| `PROMPT_PREFIX` | Fixed prompt prefix that standardizes the style of every generation request |
| `GENERATOR_SEED` | Global fixed seed (default 42); guarantees identical images on every re-run |
| `clean_title()` | Fixes encoding and strips trailing publisher/section suffixes from titles |
| `sanitize_prompt()` | Removes quoted substrings and leading dashes to keep prompts clean |
| `has_text()` | Uses EasyOCR to detect text in an image; confidence > `threshold` (0.5) marks it as containing text and rejects it |
| `score_standard_clip()` | Returns the standard CLIP logit (image-text alignment score) |
| `score_cft_clip()` | Returns the CFT-CLIP cosine similarity (-1 to 1); higher means a better image-text match |
| `get_image_embedding()` | Returns the L2-normalized CFT-CLIP image embedding |
| `get_text_embedding()` | Returns the L2-normalized CFT-CLIP text embedding |
| `generate_thumbnail()` | Main generation function: generates n candidates with `pipe` (SDXL-Turbo), filters out candidates containing text, and selects the best image by CFT-CLIP score |


In [ ]:
# @title pipeline

import re
import ftfy
import numpy as np
from PIL import Image, ImageOps

PROMPT_PREFIX = "news image thumbnail, illustration, vector art, "

# Fixed base seed for SDXL-Turbo - ensures reproducible generation across runs
# See: https://huggingface.co/docs/diffusers/v0.12.0/en/using-diffusers/reproducibility
GENERATOR_SEED = 42

# Patterns for trailing publisher/section suffixes to strip
_PIPE_SUFFIX  = re.compile(r'\s*\|.*$')
_DASH_TRAILER = re.compile(
    r'\s*-\s+('
    r'(News|Recap|Roundup|Update|Report|Review|Preview|Podcast|Video|Photo|Photos|Gallery)'
    r'|((January|February|March|April|May|June|July|August|September|October|November|December)\s+\d)'
    r'|(Week|Day|Season|Episode|Vol|No)\s+\d'
    r').*$',
    re.IGNORECASE
)

def clean_title(raw: str) -> str:
    """Fix encoding, strip trailing publisher suffixes, collapse whitespace."""
    t = ftfy.fix_text(str(raw))
    t = _PIPE_SUFFIX.sub('', t)
    t = _DASH_TRAILER.sub('', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


def sanitize_prompt(raw_text: str) -> str:
    """Strip quoted sub-strings and leading dashes from prompt text."""
    text = re.sub(r'["\'].*?["\']', '', raw_text)
    text = re.sub(r'^.*?\-\s*', '', text)
    return text.strip()


def has_text(image: Image.Image, threshold: float = 0.5) -> bool:
    """Return True if EasyOCR detects high-confidence rendered text in the image."""
    results = reader.readtext(np.array(image))
    return any(prob > threshold and len(t.strip()) > 1 for _, t, prob in results)


def score_standard_clip(image: Image.Image, text: str) -> float:
    """Compute image-text alignment with standard OpenAI CLIP."""
    inputs = standard_clip_processor(
        text=[text], images=image, return_tensors='pt', padding=True
    ).to(DEVICE)
    with torch.no_grad():
        outputs = standard_clip_model(**inputs)
    return outputs.logits_per_image.item()


def score_cft_clip(image: Image.Image, text: str) -> float:
    """Compute image-text cosine similarity with news-specific CFT-CLIP."""
    inputs = cft_clip_processor(
        text=[text], images=image, return_tensors='pt', padding=True
    ).to(DEVICE)
    with torch.no_grad():
        outputs = cft_clip_model(**inputs)
        t = F.normalize(outputs.text_embeds,  p=2, dim=-1)
        i = F.normalize(outputs.image_embeds, p=2, dim=-1)
    return torch.matmul(t, i.t()).item()


def get_image_embedding(image: Image.Image) -> np.ndarray:
    """L2-normalised CFT-CLIP image embedding (used for CFT scoring fallback)."""
    # CFT-CLIP does not expose get_image_features(); use a full forward pass
    # with a dummy empty string to obtain image_embeds from the joint output.
    inputs = cft_clip_processor(
        text=[""], images=image, return_tensors="pt", padding=True
    ).to(DEVICE)
    with torch.no_grad():
        outputs = cft_clip_model(**inputs)
        image_features = F.normalize(outputs.image_embeds, p=2, dim=-1)
    return image_features.cpu().numpy().squeeze()


def get_text_embedding(text: str) -> np.ndarray:
    """
    L2-normalised CFT-CLIP text embedding for a news headline.
    """
    # Pass a neutral grey dummy image; only text_embeds is used from the output.
    dummy_img = Image.new("RGB", (224, 224), color=(128, 128, 128))
    inputs = cft_clip_processor(
        text=[text], images=dummy_img, return_tensors="pt", padding=True
    ).to(DEVICE)
    with torch.no_grad():
        outputs = cft_clip_model(**inputs)
        text_features = F.normalize(outputs.text_embeds, p=2, dim=-1)
    return text_features.cpu().numpy().squeeze()


def generate_thumbnail(headline: str, n: int = 5):
    """
    Generate n candidates for *headline* with SDXL-Turbo (1-step).

    Each candidate uses a *fixed, reproducible seed* derived from
    GENERATOR_SEED so that re-running the notebook always produces the
    same images (HuggingFace diffusers reproducibility pattern).

    Rejects candidates that contain visible text (OCR filter).
    Selects the best candidate by CFT-CLIP cosine similarity score.

    Returns
    -------
    best_image     : PIL.Image (460x260)
    best_embedding : np.ndarray  CFT-CLIP image vector of best_image
    best_cft_score : float       CFT-CLIP score of best_image vs headline
    history        : list of dicts with keys image, std_score, cft_score, rejected
    """
    clean  = sanitize_prompt(headline)
    prompt = PROMPT_PREFIX + clean

    best_image, best_embedding, best_cft, fallback = None, None, -float('inf'), None
    history = []

    for i in range(n):
        # Fixed per-candidate seed: GENERATOR_SEED + i keeps each slot
        # reproducible while still producing n distinct images.
        generator = torch.Generator(device=DEVICE).manual_seed(GENERATOR_SEED + i)

        out = pipe(
            prompt=prompt,
            num_inference_steps=1,
            guidance_scale=0.0,
            height=512,
            width=512,
            generator=generator,
        )
        img = ImageOps.fit(out.images[0], (460, 260), method=Image.Resampling.LANCZOS)
        if fallback is None:
            fallback = img

        rejected  = has_text(img)
        std_score = cft_score = -1.0
        embedding = None

        if not rejected:
            std_score = score_standard_clip(img, clean)
            cft_score = score_cft_clip(img, clean)
            embedding = get_image_embedding(img)
            if cft_score > best_cft:
                best_cft       = cft_score
                best_image     = img
                best_embedding = embedding

        history.append({
            'image':     img,
            'std_score': std_score,
            'cft_score': cft_score,
            'rejected':  rejected,
        })

    # Fallback: if all candidates were rejected, use the first generated image
    if best_image is None:
        best_image     = fallback
        best_embedding = get_image_embedding(fallback)
        best_cft       = score_cft_clip(fallback, clean)

    return best_image, best_embedding, best_cft, history


## Step 1: Generate the best thumbnail for every test article

**Purpose:** Read the test CSV and run best-of-5 thumbnail generation for each article, saving the selected thumbnail to disk.

**Key variables:**

| Variable | Description |
|---|---|
| `N_CANDIDATES` | Number of candidate images per article (5) |
| `OUT_DIR` | Directory where thumbnails are saved (`thumbnails_best/`) |
| `df_test` | The test article DataFrame loaded from CSV |
| `cft_scores` | List of the best CFT-CLIP score per article |


In [ ]:
# @title Step 1 - Generate best-of-5 thumbnails for all test articles

import os
import pandas as pd

N_CANDIDATES = 5          # best-of-N per article (as required)
OUT_DIR      = "thumbnails_best"
os.makedirs(OUT_DIR, exist_ok=True)

# Load test CSV - update filename path if needed in Colab
df_test = pd.read_csv("news_articles_test.csv", encoding="latin1")
print(f'Loaded {len(df_test)} articles from news_articles_test.csv')

# Accumulators
article_ids     = []   # article ID per row
headlines_clean = []   # cleaned headline text
cft_scores      = []   # best-of-5 CFT-CLIP score per article

for idx, row in df_test.iterrows():
    article_id = row['article_id']
    raw_title  = row['article_title']
    headline   = clean_title(raw_title)

    print(f"[{idx+1}/{len(df_test)}] ID={article_id}  '{headline[:60]}'")

    # Generate 5 candidates; pick the best by CFT-CLIP score
    best_img, _, best_score, _ = generate_thumbnail(headline, n=N_CANDIDATES)

    # Save the selected thumbnail to disk
    fname = f'{article_id}.png'
    best_img.save(os.path.join(OUT_DIR, fname), 'PNG')

    article_ids.append(article_id)
    headlines_clean.append(headline)
    cft_scores.append(best_score)

print(f'\nDone! Generated {len(article_ids)} thumbnails.')
print(f'CFT-CLIP score range : [{min(cft_scores):.4f}, {max(cft_scores):.4f}]')


## Step 2: Save the results CSV and package the submission ZIP

**Purpose:** Collect each article's best CFT-CLIP score and thumbnail path into a CSV, then package the generated thumbnails into a submission ZIP.

**Key variables:**

| Variable | Description |
|---|---|
| `GROUP_NAME` | Team name, used as the ZIP filename prefix |
| `results_df` | Results DataFrame (one row per article) |
| `ZIP_PATH` | Output ZIP path |


In [ ]:
# @title Step 2 - Save results CSV + submission ZIP

import os, zipfile
import pandas as pd

GROUP_NAME = 'TEAM_A++'

# Results dataframe: article + best CFT-CLIP score + thumbnail path
results_df = pd.DataFrame({
    'article_id':     article_ids,
    'headline':       headlines_clean,
    'cft_score':      cft_scores,
    'thumbnail_path': [os.path.join('thumbnails_best', f'{aid}.png') for aid in article_ids],
})
results_df.to_csv('generation_results.csv', index=False)
print('Saved generation_results.csv')
print(results_df.head(10).to_string())

# Package the thumbnails into a submission ZIP
ZIP_PATH = f'{GROUP_NAME}_Submission.zip'
with zipfile.ZipFile(ZIP_PATH, 'w') as zf:
    for fname in sorted(os.listdir('thumbnails_best')):
        zf.write(os.path.join('thumbnails_best', fname),
                 arcname=f'thumbnails_best/{fname}')
    if os.path.exists('generation_results.csv'):
        zf.write('generation_results.csv')

print(f'Zipped {len(os.listdir("thumbnails_best"))} thumbnails -> {ZIP_PATH}')

# Download in Colab
from google.colab import files
files.download(ZIP_PATH)
